# 多 Agent 通信：Codex-style 与 Claude-style

共同核心：

```text
父 Agent --message--> mailbox/inbox --回合边界--> 子 Agent
父 Agent --wait/output--> 子 Agent 结果
```

Claude Code 更常把这个过程表达为 `Task` / `SendMessage` / `TaskOutput`；
Codex 更常表达为 `spawn_agent` / `send_input` / `wait_agent`。

真实生产实现里还有 thread_id、事件、预算、取消和资源清理；下面的最小运行时把这些机制压缩成一个可读模型。

In [ ]:
from pathlib import Path
import json, tempfile
from langchain_core.messages import AIMessage
from agent_runtime import SubagentRuntime
from claude_harness.events import EventBus
from claude_harness.multi_agent import build_task_tools
from mini_codex.multi_agent import build_codex_agent_tools

class FakeModel:
    def bind_tools(self, tools, **kwargs):
        return self
    def invoke(self, messages):
        return AIMessage(content="子 Agent 已完成只读探索。")

workspace = Path(tempfile.mkdtemp(prefix="multi_agent_nb_"))
bus = EventBus(workspace / "trace", console=False)


## 1. 共享 mailbox runtime

这段代码同时展示 Codex 和 Claude 都需要的底层动作：spawn、send、wait、list、close。

In [ ]:
runtime = SubagentRuntime(FakeModel(), [], event_bus=bus, max_steps=2)
spawned = runtime.spawn("explore", "检查 calculator.py", "只读探索")
agent_id = spawned["agent_id"]
print("spawned:", spawned)
print("queued:", runtime.send_input(agent_id, "请补充文件路径"))
print("result:", runtime.wait(agent_id, timeout=5))
print("agents:", runtime.list_agents())
print("close:", runtime.close(agent_id))


## 2. Codex 风格工具

`spawn_agent` 返回 agent_id，`send_input` 进入 mailbox，`wait_agent` 读取最终状态。

In [ ]:
codex_tools = {t.name: t for t in build_codex_agent_tools(FakeModel(), [], bus)}
spawn_result = json.loads(codex_tools["spawn_agent"].invoke({"description": "检查 calculator.py", "role": "explore"}))
print("spawn_agent:", spawn_result)
print("wait_agent:", codex_tools["wait_agent"].invoke({"agent_id": spawn_result["agent_id"], "timeout_sec": 5}))


## 3. Claude 风格工具

`Task` 返回 agent_id，`SendMessage` 投递 pending message，`TaskOutput` 等待结果。

In [ ]:
claude_tools = {t.name: t for t in build_task_tools(FakeModel(), [], bus)}
task_result = json.loads(claude_tools["Task"].invoke({"description": "检查 calculator.py", "subagent_type": "explore"}))
print("Task:", task_result)
print("SendMessage:", claude_tools["SendMessage"].invoke({"agent_id": task_result["agent_id"], "message": "补充文件路径"}))
print("TaskOutput:", claude_tools["TaskOutput"].invoke({"agent_id": task_result["agent_id"], "timeout_sec": 5}))


## 4. 对应源码

- Codex：`codex-rs/core/src/agent/control/mailbox.rs`、`spawn.rs`、`multi_agents_v2/`
- Claude Code：`src/tools/AgentTool/`、`src/tools/SendMessageTool/`、`src/tools/TaskOutputTool/`
- 本项目：`agent_runtime/runtime.py`、`claude_harness/multi_agent.py`、`mini_codex/multi_agent.py`
- 详细说明：`docs/multi-agent-communication.md`
